# 00 — Inventaire des données : Inondations France

Ce notebook recense et teste l'accès à toutes les sources de données nécessaires pour l'analyse du risque inondation en France, organisées en 5 catégories :

| Catégorie | Sources principales |
|-----------|--------------------|
| 🔴 **Hazard** | TRI, PPRI/GASPAR, CatNat, Vigicrues |
| 🏔️ **Terrain** | Copernicus DEM 30m, IGN RGE ALTI 1m |
| 🏘️ **Exposure** | BD TOPO, OSM, FILOSOFI, DVF, SIRENE |
| 🟡 **Vulnerability** | JRC Huizinga 2017, CLIMAAX |
| 🌧️ **Rainfall / Climat** | ERA5, SAFRAN, COMEPHORE, GPM |

**Département pilote : Hérault (34)**

## 0. Setup & Constantes

In [ ]:
# Installation des dépendances
# !pip install requests geopandas pandas matplotlib folium shapely pyproj tqdm

import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display
import json
import warnings
warnings.filterwarnings('ignore')

# ─── Constantes département pilote ───────────────────────────────────────────
DEPT_CODE   = "34"                        # Hérault
DEPT_NAME   = "Hérault"
REGION_CODE = "76"                        # Occitanie

# Bounding box Hérault [lon_min, lat_min, lon_max, lat_max] — WGS84
BBOX = [2.85, 43.21, 3.98, 43.95]
BBOX_STR = f"{BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]}"

# Villes TRI dans l'Hérault
TRI_HERAULT = ["Montpellier", "Béziers"]

print(f"✅ Département : {DEPT_NAME} ({DEPT_CODE})")
print(f"📦 Bounding box : {BBOX}")

---
## 1. 🔴 HAZARD — Données d'aléa inondation

### 1.1 Arrêtés Catastrophes Naturelles (CatNat / GASPAR)

- **Source** : data.gouv.fr / Géorisques  
- **Contenu** : Historique des arrêtés CatNat par commune depuis 1982  
- **Intérêt** : Fréquence des événements passés, validation modèles  
- **API** : `https://www.georisques.gouv.fr/api/v1/`

In [ ]:
# ─── 1.1 CatNat — API Géorisques ─────────────────────────────────────────────
BASE_GEORISQUES = "https://www.georisques.gouv.fr/api/v1"

url_catnat = f"{BASE_GEORISQUES}/gaspar/catnat"
params = {
    "code_dep": DEPT_CODE,
    "type_catnat": "INNOND",   # inondations uniquement
    "page": 1,
    "page_size": 10
}

r = requests.get(url_catnat, params=params, timeout=15)
print(f"Status : {r.status_code}")

if r.status_code == 200:
    data = r.json()
    print(f"Nombre total d'arrêtés inondation dans le {DEPT_NAME} : {data.get('total', 'N/A')}")
    if data.get('data'):
        df_catnat = pd.DataFrame(data['data'])
        print(f"Colonnes disponibles : {list(df_catnat.columns)}")
        display(df_catnat.head(5))
else:
    print(f"Erreur : {r.text[:200]}")

### 1.2 TRI — Territoires à Risque Important d'Inondation (Cycle 2, 2020)

- **Source** : Géorisques / Directive Inondation 2007/60/CE  
- **Contenu** : Zones inondables pour 3 scénarios (fréquent T10-30, moyen T100-300, extrême T1000+)  
- **Hérault** : TRI Montpellier, TRI Béziers  
- **Formats** : SHP téléchargeable + WFS  
- **URL download** : https://www.georisques.gouv.fr/donnees/bases-de-donnees/zonages-inondation-rapportage-2020

In [ ]:
# ─── 1.2 TRI — API WFS Géorisques ────────────────────────────────────────────
# Requête WFS pour les périmètres TRI dans l'Hérault

url_wfs = "https://mapsref.brgm.fr/wxs/georisques/risques"
params_wfs = {
    "SERVICE": "WFS",
    "VERSION": "1.1.0",
    "REQUEST": "GetFeature",
    "TYPENAME": "ALEA_INOND_RAPPORTAGE",
    "OUTPUTFORMAT": "application/json",
    "CQL_FILTER": f"dep_code='{DEPT_CODE}'",
    "maxFeatures": 50
}

r_wfs = requests.get(url_wfs, params=params_wfs, timeout=20)
print(f"Status WFS TRI : {r_wfs.status_code}")

# Alternative — API Géorisques zonages inondation
url_zi = f"{BASE_GEORISQUES}/zonages_inondation"
params_zi = {
    "code_departement": DEPT_CODE,
    "page": 1,
    "page_size": 5
}
r_zi = requests.get(url_zi, params=params_zi, timeout=15)
print(f"Status API Zonages inondation : {r_zi.status_code}")
if r_zi.status_code == 200:
    zi = r_zi.json()
    print(f"Résultat : {json.dumps(zi, indent=2, ensure_ascii=False)[:500]}")

In [ ]:
# ─── 1.2b TRI — Infos téléchargement ─────────────────────────────────────────
# Le TRI national est téléchargeable en SHP depuis Géorisques
# Données cycle 2 (2020) — standard COVADIS DI v2

tri_info = {
    "Nom": "TRI Cycle 2 (2020) — Zonages Inondation Rapportage",
    "Source": "Géorisques / DGPR",
    "URL": "https://www.georisques.gouv.fr/donnees/bases-de-donnees/zonages-inondation-rapportage-2020",
    "Format": "Shapefile (SHP) + WMS + WFS",
    "Scénarios": [
        "Fréquent : T10 à T30 — forte probabilité",
        "Moyen : T100 à T300 — probabilité moyenne (directive EU)",
        "Extrême : T1000+ — faible probabilité"
    ],
    "TRI Hérault": ["TRI Montpellier", "TRI Béziers"],
    "Couches clés": [
        "SurfaceInondee_* (emprise inondée par scénario)",
        "ZoneInondee_* (hauteurs d'eau par scénario)",
        "PerimetreTRI (périmètre des TRI)"
    ],
    "CRS": "RGF93 Lambert-93 (EPSG:2154) — à reprojeter si besoin"
}

for k, v in tri_info.items():
    if isinstance(v, list):
        print(f"\n📌 {k}:")
        for item in v:
            print(f"   - {item}")
    else:
        print(f"📌 {k}: {v}")

### 1.3 PPRI — Plans de Prévention des Risques Inondation (GASPAR)

- **Source** : data.gouv.fr — dataset `536995eea3a729239d20486b`  
- **Contenu** : Communes couvertes par PPRI approuvés ou en cours  
- **Intérêt** : Zonage réglementaire, zones rouges/bleues

In [ ]:
# ─── 1.3 GASPAR — data.gouv.fr ───────────────────────────────────────────────
BASE_DATAGOUV = "https://www.data.gouv.fr/api/1"

# Récupération des ressources du dataset GASPAR
dataset_id = "536995eea3a729239d20486b"
r_gaspar = requests.get(f"{BASE_DATAGOUV}/datasets/{dataset_id}/", timeout=15)
print(f"Status GASPAR : {r_gaspar.status_code}")

if r_gaspar.status_code == 200:
    gaspar = r_gaspar.json()
    print(f"Titre : {gaspar.get('title')}")
    print(f"Organisation : {gaspar.get('organization', {}).get('name', 'N/A')}")
    print(f"\nRessources disponibles ({len(gaspar.get('resources', []))}) :")
    for res in gaspar.get('resources', [])[:8]:
        print(f"  - [{res.get('format', '?'):8s}] {res.get('title', 'N/A')[:60]}")
        print(f"           URL: {res.get('url', '')[:80]}")

### 1.4 Vigicrues — Données hydrométrie temps réel

- **Source** : Hub'Eau Hydrométrie API  
- **Contenu** : Hauteurs et débits des stations de mesure  
- **API** : `https://hubeau.eaufrance.fr/api/v1/hydrometrie/`

In [ ]:
# ─── 1.4 Hub'Eau Hydrométrie — Stations dans l'Hérault ───────────────────────
HUBEAU_BASE = "https://hubeau.eaufrance.fr/api/v1/hydrometrie"

# Stations de mesure dans le département 34
r_stations = requests.get(
    f"{HUBEAU_BASE}/referentiel/stations",
    params={"code_departement": DEPT_CODE, "size": 100, "fields": "code_station,libelle_station,longitude_station,latitude_station,libelle_cours_eau"},
    timeout=15
)
print(f"Status stations : {r_stations.status_code}")

if r_stations.status_code == 200:
    stations_data = r_stations.json()
    df_stations = pd.DataFrame(stations_data.get('data', []))
    print(f"Nombre de stations dans l'Hérault : {len(df_stations)}")
    if not df_stations.empty:
        display(df_stations[['code_station','libelle_station','libelle_cours_eau']].head(10))

In [ ]:
# ─── 1.4b Observations récentes sur une station clé (Hérault à Gignac) ───────
# Station Y3524010 = L'Hérault à Gignac

if not df_stations.empty:
    station_ex = df_stations.iloc[0]['code_station']
    r_obs = requests.get(
        f"{HUBEAU_BASE}/observations_tr",
        params={
            "code_entite": station_ex,
            "grandeur_hydro": "H",   # H = hauteur, Q = débit
            "size": 96               # ~48h si pas 30min
        },
        timeout=15
    )
    if r_obs.status_code == 200:
        obs = r_obs.json()
        df_obs = pd.DataFrame(obs.get('data', []))
        print(f"Station : {station_ex} — {len(df_obs)} observations")
        if not df_obs.empty and 'date_obs' in df_obs.columns:
            df_obs['date_obs'] = pd.to_datetime(df_obs['date_obs'])
            df_obs = df_obs.sort_values('date_obs')
            # Graphique rapide
            fig, ax = plt.subplots(figsize=(10, 3))
            ax.plot(df_obs['date_obs'], df_obs['resultat_obs'].astype(float), color='steelblue')
            ax.set_title(f"Hauteur d'eau — Station {station_ex}", fontsize=12)
            ax.set_ylabel("Hauteur (mm)")
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

---
## 2. 🏔️ TERRAIN — Modèles numériques de terrain

### 2.1 Copernicus DEM GLO-30 (30m)

- **Source** : Copernicus Land Service / ESA  
- **Résolution** : 30m (~1 arc-seconde)  
- **Accès** : API Planetary Computer (Microsoft STAC) ou téléchargement direct  
- **Licence** : Libre, attribution ESA/Copernicus

In [ ]:
# ─── 2.1 Copernicus DEM — via STAC Planetary Computer ────────────────────────
# !pip install planetary-computer pystac-client rioxarray

# Option A — Planetary Computer (STAC)
dem_stac_info = {
    "Nom": "Copernicus DEM GLO-30",
    "Source": "ESA / Copernicus Land Service",
    "Résolution": "30m (1 arc-seconde)",
    "CRS": "WGS84 (EPSG:4326)",
    "Accès STAC": "https://planetarycomputer.microsoft.com/dataset/cop-dem-glo-30",
    "Collection STAC": "cop-dem-glo-30",
    "Code accès": [
        "import planetary_computer",
        "import pystac_client",
        "catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1')",
        "items = catalog.search(collections=['cop-dem-glo-30'], bbox=BBOX)"
    ],
    "Alternative directe": "https://copernicus-dem-30m.s3.amazonaws.com/"
}

# Option B — open-elevation API (preview léger)
dem_alt_info = {
    "Nom": "IGN RGE ALTI (1m)",
    "Source": "IGN France",
    "Résolution": "1m (métropole complète)",
    "Accès": "https://geoservices.ign.fr/rgealti",
    "API WMTS/WCS": "https://wxs.ign.fr/altimetrie/geoportail/r/wcs",
    "LiDAR HD (50cm)": "https://geoservices.ign.fr/lidarhd (en cours de déploiement national)"
}

print("=== Option recommandée pour démarrer ===")
for k, v in dem_stac_info.items():
    if isinstance(v, list):
        print(f"\n📌 {k}:")
        for line in v:
            print(f"   {line}")
    else:
        print(f"📌 {k}: {v}")

print("\n=== Pour haute résolution (France) ===")
for k, v in dem_alt_info.items():
    print(f"📌 {k}: {v}")

In [ ]:
# ─── 2.1b Test téléchargement Copernicus DEM via py3dep (SRTM/3DEP) ──────────
# py3dep supporte aussi le Copernicus DEM via AWS Open Data
# !pip install py3dep

# Exemple de code — à décommenter après installation
download_code = """
import py3dep
import rioxarray

# Téléchargement Copernicus DEM 30m sur l'Hérault
dem = py3dep.get_dem(
    geometry=(BBOX[0], BBOX[1], BBOX[2], BBOX[3]),
    resolution=30,
    crs="EPSG:4326"
)

# Sauvegarde locale
dem.rio.to_raster('../data/dem_herault_30m.tif')
print(f"DEM sauvegardé : {dem.shape}, résolution {dem.rio.resolution()}")
"""

print("Code de téléchargement DEM (à décommenter) :")
print(download_code)

# Alternative : OpenTopography API (sans installation)
opentopo_url = (
    f"https://portal.opentopography.org/API/globaldem?"
    f"demtype=COP30"
    f"&south={BBOX[1]}&north={BBOX[3]}&west={BBOX[0]}&east={BBOX[2]}"
    f"&outputFormat=GTiff"
    # &API_Key=... (clé gratuite sur opentopography.org)
)
print(f"\n📡 OpenTopography URL (Copernicus DEM) :")
print(opentopo_url)

---
## 3. 🏘️ EXPOSURE — Données d'exposition

### 3.1 BD TOPO IGN — Bâtiments

- **Source** : IGN / data.gouv.fr  
- **Contenu** : Emprise, hauteur, usage des bâtiments  
- **Format** : GeoParquet (par département), SHP, WFS  
- **Licence** : Licence ouverte Etalab

In [ ]:
# ─── 3.1 BD TOPO — Accès ressources data.gouv.fr ─────────────────────────────
bdtopo_dataset_id = "61489083dc4223219e50cc35"

r_bdtopo = requests.get(f"{BASE_DATAGOUV}/datasets/{bdtopo_dataset_id}/", timeout=15)
print(f"Status BD TOPO : {r_bdtopo.status_code}")

if r_bdtopo.status_code == 200:
    bdtopo = r_bdtopo.json()
    print(f"Titre : {bdtopo.get('title')}")
    resources = bdtopo.get('resources', [])
    print(f"Nombre de ressources : {len(resources)}")
    # Filtrer les ressources pour l'Hérault
    herault_resources = [r for r in resources if '034' in r.get('title','') or 'herault' in r.get('title','').lower()]
    print(f"\nRessources Hérault (34) :")
    for res in herault_resources[:5]:
        print(f"  - [{res.get('format','?'):12s}] {res.get('title','N/A')[:70]}")
        print(f"             {res.get('url','')[:80]}")
    if not herault_resources:
        print("  (Pas de ressource Hérault trouvée, affichage des 5 premières ressources)")
        for res in resources[:5]:
            print(f"  - [{res.get('format','?'):12s}] {res.get('title','N/A')[:70]}")

### 3.2 OpenStreetMap (OSM) — Bâtiments

- **Source** : OpenStreetMap  
- **Accès Python** : `osmnx` ou Overpass API  
- **Intérêt** : Alternative mondiale à BD TOPO, attributs usage (residential, commercial...)

In [ ]:
# ─── 3.2 OSM — Overpass API ───────────────────────────────────────────────────
# !pip install osmnx

osmnx_code = """
import osmnx as ox

# Téléchargement des bâtiments OSM dans l'Hérault (ou une commune)
buildings = ox.features_from_bbox(
    bbox=(BBOX[3], BBOX[1], BBOX[2], BBOX[0]),   # N, S, E, W
    tags={'building': True}
)
print(f"Bâtiments OSM : {len(buildings)} éléments")
print(f"Attributs : {list(buildings.columns[:10])}")
"""

# Test via Overpass API (sans installation)
overpass_url = "https://overpass-api.de/api/interpreter"
query = f"""
[out:json][timeout:10][bbox:{BBOX[1]},{BBOX[0]},{BBOX[3]},{BBOX[2]}];
(
  node["building"](around:500,43.6,3.9);
  way["building"](around:500,43.6,3.9);
);
out count;
"""

r_osm = requests.post(overpass_url, data=query, timeout=20)
if r_osm.status_code == 200:
    result = r_osm.json()
    count = result.get('elements', [])
    print(f"✅ Overpass API opérationnelle")
    print(f"Test zone Montpellier (500m) : {result}")
else:
    print(f"Erreur Overpass : {r_osm.status_code}")

print("\nCode osmnx complet (à décommenter) :")
print(osmnx_code)

### 3.3 FILOSOFI — Population & revenus (INSEE)

- **Source** : INSEE — Fichier localisé social et fiscal  
- **Contenu** : Population, revenus médians, ménages — maille 200m  
- **Intérêt** : Vulnérabilité sociale, estimation pertes par classe de revenus

In [ ]:
# ─── 3.3 FILOSOFI — data.gouv.fr ─────────────────────────────────────────────
filosofi_id = "5d8e16106f44410332a79994"

r_filo = requests.get(f"{BASE_DATAGOUV}/datasets/{filosofi_id}/", timeout=15)
print(f"Status FILOSOFI : {r_filo.status_code}")

if r_filo.status_code == 200:
    filo = r_filo.json()
    print(f"Titre : {filo.get('title')}")
    print(f"Dernière mise à jour : {filo.get('last_modified', 'N/A')[:10]}")
    print(f"\nRessources disponibles :")
    for res in filo.get('resources', [])[:6]:
        print(f"  - [{res.get('format','?'):8s}] {res.get('title','N/A')[:65]}")

### 3.4 DVF — Demandes de Valeurs Foncières

- **Source** : DGFiP / data.gouv.fr  
- **Contenu** : Transactions immobilières géolocalisées  
- **Intérêt** : Valeur des biens exposés → calcul des pertes économiques potentielles

In [ ]:
# ─── 3.4 DVF Géolocalisées — API data.gouv.fr ────────────────────────────────
dvf_id = "5cc1b94a634f4165e96436c1"

r_dvf = requests.get(f"{BASE_DATAGOUV}/datasets/{dvf_id}/", timeout=15)
if r_dvf.status_code == 200:
    dvf = r_dvf.json()
    print(f"Titre : {dvf.get('title')}")
    resources = dvf.get('resources', [])
    # Trouver la ressource Hérault
    herault_dvf = [r for r in resources if '34' in r.get('title','') or 'herault' in r.get('title','').lower()]
    if herault_dvf:
        print(f"\nRessource Hérault :")
        for res in herault_dvf[:3]:
            print(f"  - {res.get('title','N/A')[:70]}")
            print(f"    URL : {res.get('url','')[:80]}")
    else:
        print(f"Ressources disponibles ({len(resources)}) :")
        for res in resources[:5]:
            print(f"  - [{res.get('format','?'):6s}] {res.get('title','N/A')[:65]}")

---
## 4. 🟡 VULNERABILITY — Courbes de dommages

### 4.1 JRC — Courbes de vulnérabilité globales (Huizinga et al. 2017)

- **Source** : Joint Research Centre (JRC) — rapport JRC105688  
- **Contenu** : Courbes fraction-dommage × hauteur d'eau pour 6 types d'actifs  
- **Référence** : https://publications.jrc.ec.europa.eu/repository/handle/JRC105688  
- **CLIMAAX** : https://handbook.climaax.eu/resources/FAQ/vulnerability_curves.html

**Principe** :  `Dommage (€) = f(hauteur_eau) × valeur_maximale_actif`

In [ ]:
# ─── 4.1 Courbes JRC Huizinga 2017 — Implémentation Python ───────────────────
import numpy as np

# Courbes de vulnérabilité JRC pour l'Europe (fraction dommage en fonction hauteur eau)
# Source : Huizinga et al. (2017), Table A2 - European damage curves
# Hauteur d'eau en mètres, fraction en [0, 1]

JRC_CURVES = {
    "residential": {
        "depth_m":    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0],
        "damage_frac":[0.00, 0.25, 0.40, 0.50, 0.60, 0.75, 0.85, 0.95, 1.00],
        "max_damage_eur_m2": 1010,   # France — valeur JRC (€/m²)
        "color": "#e74c3c"
    },
    "commercial": {
        "depth_m":    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0],
        "damage_frac":[0.00, 0.35, 0.55, 0.65, 0.72, 0.82, 0.90, 0.97, 1.00],
        "max_damage_eur_m2": 730,
        "color": "#e67e22"
    },
    "industrial": {
        "depth_m":    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0],
        "damage_frac":[0.00, 0.15, 0.27, 0.38, 0.50, 0.65, 0.78, 0.90, 1.00],
        "max_damage_eur_m2": 460,
        "color": "#9b59b6"
    },
    "agriculture": {
        "depth_m":    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0],
        "damage_frac":[0.00, 0.10, 0.20, 0.30, 0.40, 0.55, 0.65, 0.80, 1.00],
        "max_damage_eur_m2": 1.5,    # €/m² (champs)
        "color": "#27ae60"
    },
    "transport": {
        "depth_m":    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0],
        "damage_frac":[0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.55, 0.75],
        "max_damage_eur_m2": 82,
        "color": "#3498db"
    }
}

def get_damage_fraction(asset_type: str, water_depth_m: float) -> float:
    """Interpolation linéaire sur la courbe JRC pour un type d'actif et une hauteur d'eau."""
    curve = JRC_CURVES.get(asset_type)
    if curve is None:
        raise ValueError(f"Type d'actif inconnu : {asset_type}. Options : {list(JRC_CURVES.keys())}")
    return float(np.interp(water_depth_m, curve["depth_m"], curve["damage_frac"]))

def compute_damage(asset_type: str, water_depth_m: float, area_m2: float) -> dict:
    """Calcule le dommage économique estimé (€) pour un actif donné."""
    curve = JRC_CURVES[asset_type]
    frac  = get_damage_fraction(asset_type, water_depth_m)
    max_dmg = curve["max_damage_eur_m2"] * area_m2
    damage  = frac * max_dmg
    return {"asset": asset_type, "depth_m": water_depth_m, "area_m2": area_m2,
            "fraction": round(frac, 3), "damage_eur": round(damage, 0)}

# ─── Test rapide ──────────────────────────────────────────────────────────────
test_cases = [
    ("residential", 0.5, 120),
    ("residential", 1.5, 120),
    ("commercial",  1.0, 500),
    ("agriculture", 2.0, 10000),
]
print("Exemples de calcul de dommages (JRC Huizinga 2017) :")
print(f"{'Actif':<15} {'Hauteur':>8} {'Surface':>10} {'Fraction':>10} {'Dommage €':>12}")
print("-" * 60)
for asset, depth, area in test_cases:
    r = compute_damage(asset, depth, area)
    print(f"{r['asset']:<15} {r['depth_m']:>7.1f}m {r['area_m2']:>9.0f}m² {r['fraction']:>9.1%} {r['damage_eur']:>12,.0f}€")

In [ ]:
# ─── 4.1b Visualisation des courbes JRC ──────────────────────────────────────
depths = np.linspace(0, 6, 100)

fig, ax = plt.subplots(figsize=(10, 5))
for asset, curve in JRC_CURVES.items():
    fractions = [np.interp(d, curve["depth_m"], curve["damage_frac"]) for d in depths]
    ax.plot(depths, fractions, label=asset.capitalize(), color=curve["color"], linewidth=2)

ax.set_xlabel("Hauteur d'eau (m)", fontsize=11)
ax.set_ylabel("Fraction de dommage [0–1]", fontsize=11)
ax.set_title("Courbes de vulnérabilité JRC — Huizinga et al. (2017) — Europe", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 6)
ax.set_ylim(0, 1)
# Zones de scénarios TRI
ax.axvspan(0, 0.5, alpha=0.05, color='green',  label='_')
ax.axvspan(0.5, 1.5, alpha=0.05, color='orange', label='_')
ax.axvspan(1.5, 6,   alpha=0.05, color='red',    label='_')
ax.text(0.25, 0.97, 'Fréquent', ha='center', fontsize=8, color='green', transform=ax.get_xaxis_transform())
ax.text(1.0, 0.97, 'Moyen', ha='center', fontsize=8, color='orange', transform=ax.get_xaxis_transform())
ax.text(3.0, 0.97, 'Extrême', ha='center', fontsize=8, color='red', transform=ax.get_xaxis_transform())
plt.tight_layout()
plt.show()

---
## 5. 🌧️ RAINFALL / CLIMAT — Précipitations & projections

### 5.1 ERA5 — Réanalyse climatique Copernicus (ECMWF)

- **Source** : Copernicus Climate Data Store (CDS)  
- **Résolution** : ~31km, 1h, depuis 1940  
- **Variables clés** : `total_precipitation`, `2m_temperature`, `runoff`  
- **Accès** : API CDS (`cdsapi`) — gratuit avec compte

In [ ]:
# ─── 5.1 ERA5 — cdsapi ───────────────────────────────────────────────────────
# !pip install cdsapi
# Nécessite un compte sur https://cds.climate.copernicus.eu + fichier ~/.cdsapirc

era5_code = """
import cdsapi

c = cdsapi.Client()
c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': 'total_precipitation',
        'year':  [str(y) for y in range(2000, 2025)],
        'month': [f'{m:02d}' for m in range(1, 13)],
        'day':   [f'{d:02d}' for d in range(1, 32)],
        'time':  ['00:00', '06:00', '12:00', '18:00'],
        'area':  [BBOX[3], BBOX[0], BBOX[1], BBOX[2]],  # N, W, S, E
        'format': 'netcdf'
    },
    '../data/era5_precip_herault_2000_2024.nc'
)
"""

# Tableau comparatif des sources de pluie
rainfall_sources = pd.DataFrame([
    {"Source": "ERA5",       "Organisation": "ECMWF/Copernicus", "Résolution spatiale": "31 km",  "Résolution temporelle": "1h",   "Période": "1940–présent", "Accès": "API CDS (gratuit+compte)"},
    {"Source": "SAFRAN",     "Organisation": "Météo-France",     "Résolution spatiale": "8 km",   "Résolution temporelle": "1h",   "Période": "1958–présent", "Accès": "Sur demande / Portail MF"},
    {"Source": "COMEPHORE",  "Organisation": "Météo-France",     "Résolution spatiale": "1 km",   "Résolution temporelle": "1h",   "Période": "1997–présent", "Accès": "Sur demande / Portail MF"},
    {"Source": "GPM IMERG",  "Organisation": "NASA",             "Résolution spatiale": "10 km",  "Résolution temporelle": "30min", "Période": "2000–présent", "Accès": "Libre — NASA GES DISC"},
    {"Source": "CHIRPS",     "Organisation": "UCSB/FEWS NET",    "Résolution spatiale": "5 km",   "Résolution temporelle": "Daily", "Période": "1981–présent", "Accès": "Libre — UCSB"},
    {"Source": "CMIP6",      "Organisation": "WCRP",             "Résolution spatiale": "~100km", "Résolution temporelle": "Daily", "Période": "hist + 2100", "Accès": "ESGF / Copernicus C3S"},
])

display(rainfall_sources)
print("\n→ Pour démarrer : ERA5 (meilleur compromis résolution/accessibilité)")
print("→ Pour modélisation fine : COMEPHORE 1km (si accès Météo-France)")

---
## 6. 📋 Tableau récapitulatif — Inventaire complet

In [ ]:
# ─── 6. Tableau inventaire complet ───────────────────────────────────────────
inventory = pd.DataFrame([
    # HAZARD
    {"Catégorie": "Hazard",      "Couche": "TRI Cycle 2 (3 scénarios)",      "Source": "Géorisques",       "Format": "SHP/WFS",    "Résolution": "Vecteur",   "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Hazard",      "Couche": "PPRI — Zones d'aléa",            "Source": "GASPAR/Géorisques","Format": "SHP/WFS",    "Résolution": "Vecteur",   "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Hazard",      "Couche": "Arrêtés CatNat",                 "Source": "data.gouv.fr",    "Format": "CSV/GeoJSON","Résolution": "Commune",   "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    {"Catégorie": "Hazard",      "Couche": "Vigicrues / Hub'Eau (stations)", "Source": "SCHAPI",          "Format": "JSON/API",   "Résolution": "Station",   "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    {"Catégorie": "Hazard",      "Couche": "Global Flood Database (Dartmouth)","Source": "Dartmouth/JRC", "Format": "GeoTIFF",    "Résolution": "250m",      "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    {"Catégorie": "Hazard",      "Couche": "Fathom Global Flood Maps",        "Source": "Fathom",         "Format": "GeoTIFF",    "Résolution": "90m",       "Accès": "🟡 Libre (acad.)","Priorité": "⭐⭐"},
    # TERRAIN
    {"Catégorie": "Terrain",     "Couche": "Copernicus DEM GLO-30",          "Source": "ESA/Copernicus",  "Format": "GeoTIFF",    "Résolution": "30m",       "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Terrain",     "Couche": "IGN RGE ALTI",                   "Source": "IGN",             "Format": "GeoTIFF",    "Résolution": "1m",        "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Terrain",     "Couche": "LiDAR HD IGN (50cm)",            "Source": "IGN",             "Format": "LAZ/GeoTIFF","Résolution": "50cm",      "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    {"Catégorie": "Terrain",     "Couche": "GEBCO Bathymétrie",              "Source": "GEBCO",           "Format": "GeoTIFF",    "Résolution": "450m",      "Accès": "🟢 Libre",     "Priorité": "⭐ (côtier)"},
    # EXPOSURE
    {"Catégorie": "Exposure",    "Couche": "BD TOPO Bâtiments (IGN)",        "Source": "IGN/data.gouv.fr","Format": "GeoParquet", "Résolution": "Bâtiment",  "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Exposure",    "Couche": "OSM Buildings",                  "Source": "OpenStreetMap",  "Format": "GeoJSON",    "Résolution": "Bâtiment",  "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Exposure",    "Couche": "FILOSOFI (pop + revenus 200m)",  "Source": "INSEE",          "Format": "CSV/GeoTIFF","Résolution": "200m",      "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Exposure",    "Couche": "DVF Géolocalisées",              "Source": "DGFiP/data.gouv","Format": "CSV",        "Résolution": "Parcelle",  "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    {"Catégorie": "Exposure",    "Couche": "SIRENE Géolocalisé",             "Source": "INSEE",          "Format": "Parquet",    "Résolution": "Établissem.","Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    {"Catégorie": "Exposure",    "Couche": "GHSL (pop + bâtis global)",      "Source": "JRC",            "Format": "GeoTIFF",    "Résolution": "100m",      "Accès": "🟢 Libre",     "Priorité": "⭐"},
    {"Catégorie": "Exposure",    "Couche": "OCS GE (Occupation sol)",        "Source": "IGN/Cerema",     "Format": "GeoParquet", "Résolution": "Vecteur",   "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    # VULNERABILITY
    {"Catégorie": "Vulnerability","Couche": "JRC Huizinga 2017 (EU curves)", "Source": "JRC",            "Format": "CSV/Python", "Résolution": "Par type",  "Accès": "🟢 Libre",     "Priorité": "⭐⭐⭐"},
    {"Catégorie": "Vulnerability","Couche": "CLIMAAX Vulnerability Curves",  "Source": "CLIMAAX/JRC",    "Format": "YAML/CSV",   "Résolution": "Par type",  "Accès": "🟢 Open source","Priorité": "⭐⭐⭐"},
    {"Catégorie": "Vulnerability","Couche": "HAZUS (adapté Europe)",         "Source": "FEMA",           "Format": "CSV",        "Résolution": "Par type",  "Accès": "🟢 Libre",     "Priorité": "⭐⭐"},
    # RAINFALL
    {"Catégorie": "Rainfall",    "Couche": "ERA5 (précipitations)",          "Source": "ECMWF/Copernicus","Format": "NetCDF",     "Résolution": "31km / 1h", "Accès": "🟢 Libre+compte","Priorité": "⭐⭐⭐"},
    {"Catégorie": "Rainfall",    "Couche": "SAFRAN",                        "Source": "Météo-France",   "Format": "NetCDF",     "Résolution": "8km / 1h",  "Accès": "🟡 Sur demande","Priorité": "⭐⭐"},
    {"Catégorie": "Rainfall",    "Couche": "COMEPHORE (radar)",             "Source": "Météo-France",   "Format": "NetCDF",     "Résolution": "1km / 1h",  "Accès": "🟡 Sur demande","Priorité": "⭐⭐"},
    {"Catégorie": "Rainfall",    "Couche": "GPM IMERG",                     "Source": "NASA",           "Format": "HDF5/NetCDF","Résolution": "10km / 30min","Accès": "🟢 Libre",    "Priorité": "⭐⭐"},
    {"Catégorie": "Rainfall",    "Couche": "CMIP6 (projections futures)",   "Source": "WCRP/Copernicus","Format": "NetCDF",     "Résolution": "~100km",    "Accès": "🟢 Libre",     "Priorité": "⭐ (futur)"},
])

# Affichage stylisé
print(f"INVENTAIRE COMPLET — {len(inventory)} jeux de données recensés")
print("=" * 80)
for cat in inventory['Catégorie'].unique():
    subset = inventory[inventory['Catégorie'] == cat]
    print(f"\n{'=' * 40}")
    print(f" {cat.upper()} ({len(subset)} sources)")
    print(f"{'=' * 40}")
    display(subset[['Couche','Source','Format','Résolution','Accès','Priorité']].reset_index(drop=True))

In [ ]:
# ─── Sauvegarde de l'inventaire ───────────────────────────────────────────────
import os
os.makedirs('../data', exist_ok=True)
inventory.to_csv('../data/inventaire_donnees.csv', index=False)
print("✅ Inventaire sauvegardé : ../data/inventaire_donnees.csv")
print(f"\n📊 Résumé par catégorie :")
print(inventory.groupby('Catégorie').size().to_string())